In [ ]:
# Initial exploration: the primary analysis uses the 2025 session.
# Run scripts/download_data.py once first; scripts/inspect_data.py audits both sessions.
# A row index is not a trial ID or a verified acquisition timestamp.


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import glob

In [ ]:
from pathlib import Path
DATA_ROOT = Path("data/raw")
if not list(DATA_ROOT.glob("2025-12-18/*/*.json")):
    raise FileNotFoundError("Run .venv/bin/python scripts/download_data.py first")


In [ ]:
df = pd.DataFrame(json.loads(p.read_text()) for p in sorted(DATA_ROOT.glob("2025-12-18/*/*.json")))
df


In [ ]:
BALL_WIDTH = 0.79 # diameter of a basketball, in feet

def release_frame(trial):
    from scripts.inspect_data import release_frame as detect_release
    return detect_release(trial)

def plot_release_frames():
  frames = []
  for i in range(len(df)):
    frames.append(release_frame(df.loc[i]))

  plt.xlabel("shot (row in df)");
  plt.ylabel("release frame")
  plt.scatter(range(len(frames)), frames, s=8, c=df.participant_id.astype("category").cat.codes)

def find_release_frames_outliers(df, max_off_seconds=0.5, fps=None):
    rates = df.sampling_rate if fps is None else fps
    release = df.apply(release_frame, axis=1) / rates
    usual = release.groupby(df.participant_id).transform("median")
    too_far = abs(release - usual) > max_off_seconds
    return df[too_far]

def plot_shot(trial):
    frames = trial["tracking"]
    ball = np.array([f["data"]["ball"] for f in frames], float)

    for wrist_name in ["RIGHT_WRIST", "LEFT_WRIST"]:
        wrist = np.array([f["data"]["player"][wrist_name] for f in frames], float)
        plt.plot(np.linalg.norm(ball - wrist, axis=1), label=wrist_name)

    release = release_frame(trial)
    if release is not None:
        plt.axvline(release, color="red", ls="--", label=f"release estimate ({release})")

    plt.title(f"{trial['participant_id']} {trial['trial_id']}")
    plt.xlabel("frame")
    plt.ylabel("ball to wrist (ft)")
    plt.legend()
    plt.show()

In [ ]:
plot_release_frames()

In [ ]:
find_release_frames_outliers(df)

In [ ]:
plot_shot(df.loc[10]) # put any trial